In [21]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.cluster import KMeans
import skfuzzy as fuzz
from sklearn.metrics import jaccard_score

In [20]:
DATASET_PATH = r"C:\V\sem_6_notes\DL_MedicalImage_processing\WhiteBloodCellSegmentation\BCCD Dataset with mask\test"

IMG_DIR = os.path.join(DATASET_PATH, "original")
MASK_DIR = os.path.join(DATASET_PATH, "mask")

image_files = sorted(os.listdir(IMG_DIR))
mask_files = sorted(os.listdir(MASK_DIR))
print("Total Images:", len(image_files))
print("Total Mask: ", len(mask_files))

Total Images: 159
Total Mask:  159


In [19]:
def create_features(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    lab = lab.astype(np.float32) / 255.0
    
    h, w, _ = lab.shape
    
    X_color = lab.reshape(-1, 3)
    
    x_coords, y_coords = np.meshgrid(np.arange(w), np.arange(h))
    x_coords = x_coords.reshape(-1,1) / w
    y_coords = y_coords.reshape(-1,1) / h
    
    X = np.concatenate([X_color, x_coords, y_coords], axis=1)
    
    return X, h, w

**K Means Segmentation**

In [12]:
def segment_kmeans(X, h, w):
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    return labels.reshape(h,w)

**FCM Segmentation**

In [13]:
def segment_fcm(X, h, w):
    cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
        X.T, c=3, m=2, error=0.005, maxiter=500
    )
    labels = np.argmax(u, axis=0)
    return labels.reshape(h,w)

**Detect Darkest Cluster**

In [22]:
def get_nucleus_cluster(segmentation, image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    L_channel = lab[:,:,0]
    
    clusters = np.unique(segmentation)
    avg_L = []
    
    for c in clusters:
        avg_L.append(np.mean(L_channel[segmentation==c]))
    
    return clusters[np.argmin(avg_L)]

**Dice and IOU**

In [23]:
def dice_score(pred, gt):
    intersection = np.sum(pred * gt)
    return (2. * intersection) / (np.sum(pred) + np.sum(gt) + 1e-8)

def evaluate_region(pred_mask, gt_mask):
    dice = dice_score(pred_mask, gt_mask)
    iou = jaccard_score(gt_mask.flatten(), pred_mask.flatten())
    return dice, iou

**Boundary Extraction**

In [24]:
def get_boundary(mask):
    edges = cv2.Canny((mask * 255).astype(np.uint8), 100, 200)
    return (edges > 0).astype(np.uint8)

**Boundary F1**

In [25]:
def boundary_f1_score(pred_mask, gt_mask, tolerance=2):
    
    pred_boundary = get_boundary(pred_mask)
    gt_boundary = get_boundary(gt_mask)
    
    kernel = np.ones((2*tolerance+1, 2*tolerance+1), np.uint8)
    
    pred_dilated = cv2.dilate(pred_boundary, kernel)
    gt_dilated = cv2.dilate(gt_boundary, kernel)
    
    tp_precision = np.sum(pred_boundary * gt_dilated)
    precision = tp_precision / (np.sum(pred_boundary) + 1e-8)
    
    tp_recall = np.sum(gt_boundary * pred_dilated)
    recall = tp_recall / (np.sum(gt_boundary) + 1e-8)
    
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    
    return f1

In [29]:
dice_km_list = []
dice_fcm_list = []
iou_km_list = []
iou_fcm_list = []
boundary_km_list = []
boundary_fcm_list = []

for file in image_files[:30]:
    
    img_path = os.path.join(IMG_DIR, file)
    filename = os.path.splitext(file)[0]
    
    # Robust mask loading
    possible_ext = [".png", ".jpg", ".jpeg"]
    mask_path = None
    
    for ext in possible_ext:
        temp_path = os.path.join(MASK_DIR, filename + ext)
        if os.path.exists(temp_path):
            mask_path = temp_path
            break
    
    if mask_path is None:
        print("Mask not found for:", file)
        continue
    
    # Load image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (256,256))
    
    # Load mask
    gt = cv2.imread(mask_path, 0)
    gt = cv2.resize(gt, (256,256))
    gt = (gt > 127).astype(np.uint8)
    
    # Feature extraction
    X, h, w = create_features(image)
    
    # Segmentation
    seg_km = segment_kmeans(X, h, w)
    seg_fcm = segment_fcm(X, h, w)
    
    # Detect nucleus cluster
    nucleus_km = get_nucleus_cluster(seg_km, image)
    nucleus_fcm = get_nucleus_cluster(seg_fcm, image)
    
    mask_km = (seg_km == nucleus_km).astype(np.uint8)
    mask_fcm = (seg_fcm == nucleus_fcm).astype(np.uint8)
    
    # Region metrics
    dice_km, iou_km = evaluate_region(mask_km, gt)
    dice_fcm, iou_fcm = evaluate_region(mask_fcm, gt)
    
    # Boundary metric
    bf_km = boundary_f1_score(mask_km, gt, tolerance=2)
    bf_fcm = boundary_f1_score(mask_fcm, gt, tolerance=2)
    
    dice_km_list.append(dice_km)
    dice_fcm_list.append(dice_fcm)
    iou_km_list.append(iou_km)
    iou_fcm_list.append(iou_fcm)
    boundary_km_list.append(bf_km)
    boundary_fcm_list.append(bf_fcm)

In [31]:
print("===== FINAL RESULTS =====")

#print("KMeans Dice:", np.mean(dice_km_list))
#print("FCM Dice:", np.mean(dice_fcm_list))

#print("KMeans IoU:", np.mean(iou_km_list))
#print("FCM IoU:", np.mean(iou_fcm_list))

print("KMeans Boundary F1:", np.mean(boundary_km_list))
print("FCM Boundary F1:", np.mean(boundary_fcm_list))

===== FINAL RESULTS =====
KMeans Boundary F1: 0.48813276166852765
FCM Boundary F1: 0.4836981205614279
